# CrewAI

**Module:** 11-agent-frameworks

**Notebook:** `03-crewai.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **CrewAI Overview** with clear contracts and failure modes
- Explain and apply **Agents** with clear contracts and failure modes
- Explain and apply **Tasks** with clear contracts and failure modes
- Explain and apply **Crews & Processes** with clear contracts and failure modes
- Explain and apply **Tools in Crews** with clear contracts and failure modes
- Explain and apply **When CrewAI Fits** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — CrewAI

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **CrewAI Overview**
2. **Agents**
3. **Tasks**
4. **Crews & Processes**
5. **Tools in Crews**
6. **When CrewAI Fits**

Read top-to-bottom once, then revisit weak spots with the exercises.


## CrewAI Overview

### Definition
**CrewAI Overview** is a core building block in 03-crewai within agent frameworks. Treat it as a runtime for policies and state—not a substitute for product judgment: something you can name, version, test, and operate.

### Why it matters
In agent frameworks, weak designs around CrewAI Overview typically surface as choosing framework theater over a clear control loop. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For CrewAI Overview: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like graphs, crews, handoffs, and typed agent results.

### Intuition
Explain CrewAI Overview as a runtime for policies and state—not a substitute for product judgment. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating CrewAI Overview as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for CrewAI Overview
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agent frameworks: choosing framework theater over a clear control loop

### When to use
Use CrewAI Overview when your product path depends on this concern in agent frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does CrewAI Overview improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "CrewAI Overview" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "CrewAI Overview"
    notebook: str = "03-crewai"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


In [ ]:
# Demo: decision table for applying "CrewAI Overview"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_crewai_overv", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


## Agents

### Definition
**Agents** is a core building block in 03-crewai within agent frameworks. Treat it as a runtime for policies and state—not a substitute for product judgment: something you can name, version, test, and operate.

### Why it matters
In agent frameworks, weak designs around Agents typically surface as choosing framework theater over a clear control loop. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Agents: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like graphs, crews, handoffs, and typed agent results.

### Intuition
Explain Agents as a runtime for policies and state—not a substitute for product judgment. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Agents as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Agents
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agent frameworks: choosing framework theater over a clear control loop

### When to use
Use Agents when your product path depends on this concern in agent frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Agents" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Agents"
    notebook: str = "03-crewai"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


### Worked scenario — Agents

**Situation:** A team wants to productionize a feature involving **Agents**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Tasks

### Definition
**Tasks** is a core building block in 03-crewai within agent frameworks. Treat it as a runtime for policies and state—not a substitute for product judgment: something you can name, version, test, and operate.

### Why it matters
In agent frameworks, weak designs around Tasks typically surface as choosing framework theater over a clear control loop. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Tasks: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like graphs, crews, handoffs, and typed agent results.

### Intuition
Explain Tasks as a runtime for policies and state—not a substitute for product judgment. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Tasks as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Tasks
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agent frameworks: choosing framework theater over a clear control loop

### When to use
Use Tasks when your product path depends on this concern in agent frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Tasks" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Tasks"
    notebook: str = "03-crewai"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Tasks"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Tasks"}
strong = {"definition": "Tasks", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Tasks"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Tasks", "passed": len(checks)-len(failed), "failed": failed})


## Crews & Processes

### Definition
**Crews & Processes** is a core building block in 03-crewai within agent frameworks. Treat it as a runtime for policies and state—not a substitute for product judgment: something you can name, version, test, and operate.

### Why it matters
In agent frameworks, weak designs around Crews & Processes typically surface as choosing framework theater over a clear control loop. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Crews & Processes: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like graphs, crews, handoffs, and typed agent results.

### Intuition
Explain Crews & Processes as a runtime for policies and state—not a substitute for product judgment. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Crews & Processes as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Crews & Processes
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agent frameworks: choosing framework theater over a clear control loop

### When to use
Use Crews & Processes when your product path depends on this concern in agent frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Crews & Processes" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Crews & Processes"
    notebook: str = "03-crewai"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


### Worked scenario — Crews & Processes

**Situation:** A team wants to productionize a feature involving **Crews & Processes**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Tools in Crews

### Definition
**Tools in Crews** is a core building block in 03-crewai within agent frameworks. Treat it as a runtime for policies and state—not a substitute for product judgment: something you can name, version, test, and operate.

### Why it matters
In agent frameworks, weak designs around Tools in Crews typically surface as choosing framework theater over a clear control loop. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Tools in Crews: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like graphs, crews, handoffs, and typed agent results.

### Intuition
Explain Tools in Crews as a runtime for policies and state—not a substitute for product judgment. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Tools in Crews as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Tools in Crews
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agent frameworks: choosing framework theater over a clear control loop

### When to use
Use Tools in Crews when your product path depends on this concern in agent frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Tools in Crews" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Tools in Crews"
    notebook: str = "03-crewai"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


## When CrewAI Fits

### Definition
**When CrewAI Fits** is a core building block in 03-crewai within agent frameworks. Treat it as a runtime for policies and state—not a substitute for product judgment: something you can name, version, test, and operate.

### Why it matters
In agent frameworks, weak designs around When CrewAI Fits typically surface as choosing framework theater over a clear control loop. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For When CrewAI Fits: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like graphs, crews, handoffs, and typed agent results.

### Intuition
Explain When CrewAI Fits as a runtime for policies and state—not a substitute for product judgment. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating When CrewAI Fits as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for When CrewAI Fits
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agent frameworks: choosing framework theater over a clear control loop

### When to use
Use When CrewAI Fits when your product path depends on this concern in agent frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "When CrewAI Fits" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "When CrewAI Fits"
    notebook: str = "03-crewai"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


### Worked scenario — When CrewAI Fits

**Situation:** A team wants to productionize a feature involving **When CrewAI Fits**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **CrewAI**.

| Topic | Do | Don't |
|-------|----|-------|
| CrewAI Overview | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Agents | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Tasks | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Crews & Processes | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Tools in Crews | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| When CrewAI Fits | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| CrewAI Overview | Key concept covered in this notebook; see its section for definition and pitfalls |
| Agents | Key concept covered in this notebook; see its section for definition and pitfalls |
| Tasks | Key concept covered in this notebook; see its section for definition and pitfalls |
| Crews & Processes | Key concept covered in this notebook; see its section for definition and pitfalls |
| Tools in Crews | Key concept covered in this notebook; see its section for definition and pitfalls |
| When CrewAI Fits | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **CrewAI** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **11-agent-frameworks**.


## Try It Yourself

1. Implement a failing test/fixture for **CrewAI Overview**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Agents**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Tasks**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Crews & Processes**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Tools in Crews**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
